# Phân chia Tập dữ liệu Train/Validation theo Phân tầng (Stratified Train/Val Split)

Thực hiện việc phân chia tập dữ liệu huấn luyện đã được làm sạch (`data/processed/train.csv`) thành hai tập:
- **Tập huấn luyện (Train Set - 90%)**: Dùng để điều chỉnh trọng số mô hình qua phương pháp Fine-tuning (SFT) và xây dựng RAG database.
- **Tập kiểm định (Validation Set - 10%)**: Dùng để theo dõi mức độ mất mát (validation loss) trong quá trình huấn luyện nhằm phát hiện sớm hiện tượng quá khớp (overfitting).

**Mục tiêu chính:**
1. Đọc dữ liệu sạch đã khử trùng lặp.
2. Thực hiện kỹ thuật phân tách phân tầng (Stratified Split) theo cột điểm `Overall_Band`.
3. Kiểm tra tính đồng đều của phân phối nhãn điểm giữa hai tập dữ liệu.
4. Xuất các file `train.csv` (ghi đè tập train đầy đủ cũ để sẵn sàng đưa vào SFT Trainer) và `val.csv` (tập validation 10%) vào thư mục `data/processed/`.

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

## 1. Tải Dữ liệu Sạch

Đọc file `train.csv` đã xử lý từ thư mục `data/processed/`.

In [2]:
DATA_PROCESSED_DIR = "../data/processed"
cleaned_train_path = os.path.join(DATA_PROCESSED_DIR, "train.csv")

df_clean = pd.read_csv(cleaned_train_path)
print(f"Đã load tập dữ liệu sạch: {df_clean.shape[0]} dòng, {df_clean.shape[1]} cột")

Đã load tập dữ liệu sạch: 8289 dòng, 17 cột


## 2. Chiến lược Phân tầng (Stratification Strategy)

### Tại sao cần dùng Phân tầng (Stratification)?
Trong IELTS Writing, phân phối điểm số thường không đồng đều. Đa số thí sinh đạt điểm ở dải trung bình (5.5 - 6.5), trong khi các dải điểm cực đoan như Band 2.0 hoặc Band 9.0 xuất hiện với tần suất cực kỳ thấp (ví dụ: Band 2.0 chỉ có 9 mẫu).

Kỹ thuật **Stratified Split** giải quyết vấn đề này bằng cách chia dữ liệu sao cho **tỷ lệ phần trăm các mức điểm trong tập Train và tập Val giống hệt nhau** so với phân phối ban đầu.

In [4]:
# Hiển thị phân bố số lượng mẫu của từng band trước khi chia
print("Phân bố số lượng mẫu theo Overall_Band gốc:")
print(df_clean['Overall_Band'].value_counts().sort_index())

Phân bố số lượng mẫu theo Overall_Band gốc:
Overall_Band
2.0       7
2.5      10
3.0     205
3.5     256
4.0     492
4.5     556
5.0     885
5.5     796
6.0     951
6.5     982
7.0    1145
7.5     946
8.0     604
8.5     363
9.0      91
Name: count, dtype: int64


## 3. Thực hiện Phân chia Dữ liệu Train/Val

Sử dụng hàm `train_test_split` của Scikit-learn với tham số `stratify=df_clean['Overall_Band']` để thực hiện phân tầng.

In [5]:
# Cấu hình các tham số phân chia
VAL_SIZE = 0.10  # 10% làm tập kiểm định, 90% làm tập huấn luyện
RANDOM_SEED = 42 # Đảm bảo khả năng tái lặp kết quả

# Kiểm tra và loại bỏ các nhóm điểm chỉ xuất hiện duy nhất 1 lần (nếu có) để tránh lỗi phân tầng
band_counts = df_clean['Overall_Band'].value_counts()
rare_bands = band_counts[band_counts < 2].index

if len(rare_bands) > 0:
    print(f"Cảnh báo: Loại bỏ các band điểm có ít hơn 2 mẫu để tránh lỗi phân tầng: {list(rare_bands)}")
    df_clean = df_clean[~df_clean['Overall_Band'].isin(rare_bands)]

# Thực hiện chia tách phân tầng
df_train_split, df_val_split = train_test_split(
    df_clean,
    test_size=VAL_SIZE,
    random_state=RANDOM_SEED,
    stratify=df_clean['Overall_Band']
)

print(f"Kích thước sau phân chia:")
print(f"- Tập Train mới (90%): {df_train_split.shape}")
print(f"- Tập Validation (10%): {df_val_split.shape}")

Kích thước sau phân chia:
- Tập Train mới (90%): (7460, 17)
- Tập Validation (10%): (829, 17)


## 4. Kiểm tra Tính Đồng đều Phân phối Điểm số

Tính toán tỷ lệ phần trăm (%) của từng band điểm trong tập Train và tập Val để chứng minh tính hiệu quả của chiến lược phân tầng.

In [7]:
# Tính toán tỷ lệ phần trăm
train_pct = df_train_split['Overall_Band'].value_counts(normalize=True) * 100
val_pct = df_val_split['Overall_Band'].value_counts(normalize=True) * 100

# Gộp kết quả thành bảng so sánh
compare_df = pd.DataFrame({
    'Train (%)': train_pct,
    'Val (%)': val_pct
}).sort_index()

# Thêm cột sai lệch tuyệt đối
compare_df['Sai lệch (Abs Diff %)'] = (compare_df['Train (%)'] - compare_df['Val (%)']).abs()

print("=== BẢNG SO SÁNH PHÂN PHỐI BAND ĐIỂM GIỮA TRAIN VÀ VAL ===")
print(compare_df.round(3))

# Kiểm tra xem sai lệch lớn nhất có vượt quá 0.5% không
max_diff = compare_df['Sai lệch (Abs Diff %)'].max()
print(f"\nSai lệch phân phối lớn nhất: {max_diff:.4f}%")
if max_diff < 0.5:
    print("✔ ĐẠT YÊU CẦU: Phân phối điểm số cực kỳ đồng đều giữa hai tập.")
else:
    print("❌ CẢNH BÁO: Sai lệch lớn, vui lòng kiểm tra lại dữ liệu.")

=== BẢNG SO SÁNH PHÂN PHỐI BAND ĐIỂM GIỮA TRAIN VÀ VAL ===
              Train (%)  Val (%)  Sai lệch (Abs Diff %)
Overall_Band                                           
2.0               0.080    0.121                  0.040
2.5               0.121    0.121                  0.000
3.0               2.480    2.413                  0.067
3.5               3.083    3.136                  0.053
4.0               5.938    5.911                  0.028
4.5               6.702    6.755                  0.053
5.0              10.684   10.615                  0.068
5.5               9.598    9.650                  0.052
6.0              11.475   11.460                  0.015
6.5              11.850   11.821                  0.028
7.0              13.807   13.872                  0.065
7.5              11.408   11.460                  0.052
8.0               7.292    7.238                  0.055
8.5               4.383    4.343                  0.041
9.0               1.099    1.086             

## 5. Xuất Các Tập Dữ liệu

Lưu các dataframe đã được phân chia thành tệp `train.csv` (ghi đè tập train đầy đủ cũ để sẵn sàng đưa vào SFT Trainer) và `val.csv` vào `data/processed/`.

In [8]:
train_out_path = os.path.join(DATA_PROCESSED_DIR, "train.csv")
val_out_path = os.path.join(DATA_PROCESSED_DIR, "val.csv")

# Xuất dữ liệu
df_train_split.to_csv(train_out_path, index=False)
df_val_split.to_csv(val_out_path, index=False)

print("Đã lưu các file phân chia dữ liệu thành công:")
print(f"- {os.path.abspath(train_out_path)} ({df_train_split.shape[0]} dòng)")
print(f"- {os.path.abspath(val_out_path)} ({df_val_split.shape[0]} dòng)")

Đã lưu các file phân chia dữ liệu thành công:
- t:\5 - Summer 2026\AES_LLM\data\processed\train.csv (7460 dòng)
- t:\5 - Summer 2026\AES_LLM\data\processed\val.csv (829 dòng)
